# Model Maxxing v0.4 — Hito A: selección del par 1.5B (Kaggle)

Ejecuta **Save Version → Save & Run All** (no hace falta dejar el navegador abierto).
Requisitos: Accelerator = GPU (T4 o P100) e Internet = On.
Al terminar, descarga `selection_kaggle.zip` de la pestaña **Output**.

- Repo en `/tmp` para que los modelos (9 GB) no se guarden como output.
- Sin Docker: los tests se ejecutan en subprocesos con límites (VM desechable), `MM_SANDBOX=subprocess`.
- Todo (NLL + pass@1 de las 6 configuraciones) se rehace aquí: un solo entorno para todas las comparaciones.

In [ ]:
!nvidia-smi
%cd /tmp
!rm -rf modelmaxxing-prev && git clone -q https://github.com/RubenGGBC/modelmaxxing-prev.git
%cd /tmp/modelmaxxing-prev
!git log --oneline -1
!pip install -q -e . "transformers==5.22.2" "datasets==5.1.0" 2>&1 | tail -3

In [ ]:
!python scripts/download_models.py v04 2>&1 | grep -v -e Warn -e Fetching

In [ ]:
import torch
DTYPE = "bf16" if torch.cuda.get_device_capability()[0] >= 8 else "fp16"   # T4/P100 no tienen bf16 nativo
print(torch.cuda.get_device_name(0), DTYPE, torch.__version__)

In [ ]:
!PYTHONPATH=src MM_SANDBOX=subprocess PYTHONIOENCODING=utf-8 python -m pytest -q 2>&1 | tail -3

In [ ]:
!PYTHONPATH=src MM_SANDBOX=subprocess PYTHONIOENCODING=utf-8 python scripts/select_models.py all     --out results/experiments/maxxing_v04/selection_kaggle --batch-size 16 --dtype {DTYPE} 2>&1     | grep -v -e Warn -e "Loading weights" -e "it/s" 

In [ ]:
!cd /tmp/modelmaxxing-prev && zip -qr /kaggle/working/selection_kaggle.zip results/experiments/maxxing_v04/selection_kaggle
!ls -la /kaggle/working